# Results Notebook

This Notebook contains the code for generating all figures and tables in the paper, using raw results from the experimaestro workspace.

# Imports and Utils

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# use ACM style

FONT_SIZE = 14

# Set font for publication quality
# plt.rcParams['font.family'] = 'Times New Roman'
# 1. Configure Matplotlib for ACM Style
plt.rcParams.update(
    {
        "text.usetex": False,  # Set to True if you have LaTeX installed on your system
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
        "font.size": 10,  # Standard ACM body text size is ~9-10pt
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "figure.titlesize": 12,
    }
)
plt.style.use("ggplot")


# default_workspace_path
# ws = get_workspace().path
# xps_root = ws / "experiments"

xps_root = Path("/Users/victor/code/experiments/JZ")


def get_last_xp(xp_id, force_date=None) -> str:
    dates = list((xps_root / xp_id).glob("*"))
    print(f"Found dates for {xp_id}: {[d.name for d in dates]}")

    # parse and get most recent date
    if force_date is not None:
        forced_date_path = xps_root / xp_id / force_date
        if forced_date_path in dates:
            most_recent_date = forced_date_path
            print(f"Using forced date for {xp_id}: {most_recent_date.name}")
            return most_recent_date
        else:
            print(f"Forced date {force_date} not found for xp {xp_id}")

    dates = [d for d in dates if re.match(r"\d+_\d+", d.name)]

    if len(dates) == 0:
        print(f"No valid dates found for xp {xp_id}, returning dry-run")
        return xps_root / xp_id / "dry-run"

    most_recent_date = max(dates, key=lambda d: d.name)

    print(f"Most recent date for {xp_id}: {most_recent_date.name}")

    return most_recent_date


def get_results(xp_path) -> pd.DataFrame:
    path = xp_path / "results"
    csv_files = list(path.glob("results.csv"))

    if len(csv_files):
        return pd.read_csv(csv_files[0])
    else:
        raise ValueError(f"no csv files found in {path}")


def process_results(results) -> pd.DataFrame:
    labels1 = results.iloc[0]
    labels2 = results.iloc[1]
    labels = [
        f"{l1}{'_' + str(l2) if str(l2) != 'nan' else ''}"
        for l1, l2 in zip(labels1, labels2)
    ]
    labels[0] = "dataset"
    results.columns = labels
    results = results.iloc[2:]
    # Select columns containing 'mean' or 'var' (case-insensitive)
    cols_to_convert = results.filter(regex="(?i)mean|var").columns

    # Convert those columns to numeric, turning errors (like 'None' strings) into NaN
    results[cols_to_convert] = results[cols_to_convert].apply(
        pd.to_numeric, errors="coerce"
    )

    return results


def get_aggregated_stats(results, group_cols, metrics):
    # 3. Ensure numeric types
    all_cols = []
    for m in metrics:
        all_cols.extend([f"{m}_mean", f"{m}_var"])

    for col in all_cols:
        results[col] = pd.to_numeric(results[col], errors="coerce")

    # 4. Compute the Simple Average of Means and Variances
    #    We do NOT add the inter-dataset variance here.
    #    We act as if the datasets are fixed strata.
    aggregated_stats = results.groupby(group_cols, dropna=False)[all_cols].mean()

    # 5. Rename columns for clarity
    #    Renaming 'AP_mean' -> 'Grand_Mean_AP' and 'AP_var' -> 'Average_Var_AP'
    rename_dict = {}
    for m in metrics:
        rename_dict[f"{m}_mean"] = f"{m}_Grand_Mean"
        rename_dict[f"{m}_var"] = f"{m}_Average_Var"

    aggregated_stats = aggregated_stats.rename(columns=rename_dict)

    return aggregated_stats
    # Add experiment label


def get_aggregated_df(df, datasets, name):
    """
    Returns a NEW DataFrame with aggregated means for specific datasets.
    Does not modify the original df.
    """
    # 1. Identify metric columns and metadata columns
    metric_cols = df.filter(regex="(?i)mean|var").columns.tolist()
    # Metadata columns are everything EXCEPT the metrics and the dataset name
    metadata_cols = [c for c in df.columns if c not in metric_cols and c != "dataset"]

    # 2. Filter rows for the specified datasets
    subset = df[df["dataset"].isin(datasets)].copy()
    subset[metric_cols] = subset[metric_cols].apply(pd.to_numeric, errors="coerce")

    # 3. Group by ALL metadata columns to preserve them
    # This keeps 'base', 'loss', 'scorer', 'arch', etc. intact
    new_df = subset.groupby(metadata_cols, as_index=False)[metric_cols].mean()

    # 4. Assign the new dataset aggregate name
    new_df["dataset"] = name

    # Reorder for readability: dataset first, then metadata, then metrics
    cols = ["dataset"] + metadata_cols + metric_cols
    return new_df[cols]


print(f"found xps in {xps_root}")
all_xps = list(xps_root.glob("*"))
for xp in sorted(all_xps):
    print(f"- {xp.name}")

### Labeling

In [ ]:
from format import loss_names, backbone_names, DATASET_TO_ABB, aggregations

ignore_losses = ["infoNCE_norm_size=True"]

# Ablations

In [ ]:
# Main table
import pandas as pd
from IPython.display import display, HTML

xps = [
    "train_mice_miniLM_bools",
]


def get_tag(tagspath: str, regex: str) -> dict:
    match = re.search(regex, tagspath)
    if match:
        return match.group(1)
    return None


raw_dfs = []
for xp in xps:
    xp_path = xps_root / xp / "dry-run"
    raw_dfs.append(process_results(get_results(xp_path)))

df = pd.concat(raw_dfs)


# Define regex patterns for tags
tags = {
    "base": r"base=(.*)_global",
    "global_cls_token": r"global_cls_token=(.*)_mask_cls_to_doc",
    "mask_query_to_cls": r"mask_query_to_cls=(.*)_loss",
    "mask_cls_to_doc": r"mask_cls_to_doc=(.*)_mask",
}

# rm row without "scorer" tag
df = df[df["scorer"].notna()]
# Split "scorer" into separate columns using the regex patterns
for tag_name, pattern in tags.items():
    df[tag_name] = df["scorer"].apply(lambda x: get_tag(x, pattern))


# Ensure the boolean columns are boolean type
bool_cols = ["global_cls_token", "mask_query_to_cls", "mask_cls_to_doc"]
for col in bool_cols:
    df[col] = df[col].map({"True": True, "False": False})

datasets = list(df["dataset"].unique())
print(f"{len(datasets)} unique datasets : {datasets}")
# print(f"unique losses : {list(df["loss"].unique())}")
# print(f"Backbones : {list(df["base"].unique())}")


display(HTML(df.iloc[0:5].to_html()))

In [ ]:
# Plot in-domain nDCG@10_mean for each combination of boolean params

import matplotlib.pyplot as plt

for dataset in ["Mean In Domain", "mean"]:
    # Filter for in-domain dataset (assuming msmarco_dev is in-domain)
    in_domain_df = df[df["dataset"] == dataset].copy()

    # Group by the boolean combinations and compute mean of nDCG@10_mean and mean of nDCG@10_var
    grouped = (
        in_domain_df.groupby(bool_cols)
        .agg(mean_nDCG=("nDCG@10_mean", "mean"), mean_var=("nDCG@10_var", "mean"))
        .reset_index()
    )

    # Compute std as sqrt of mean variance
    grouped["std_nDCG"] = np.sqrt(grouped["mean_var"])

    # Create a combination label
    grouped["combination"] = grouped[bool_cols].astype(str).agg("_".join, axis=1)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = sns.barplot(data=grouped, x="combination", y="mean_nDCG", ax=ax)
    ax.errorbar(
        x=range(len(grouped)),
        y=grouped["mean_nDCG"],
        yerr=grouped["std_nDCG"],
        fmt="none",
        c="black",
        capsize=3,
        elinewidth=1,
        alpha=0.8,
    )
    ax.set_title("Mean nDCG@10 for Boolean Parameter Combinations (In-Domain)")
    ax.set_xlabel(
        "Parameter Combination (global_cls_token_mask_query_to_cls_mask_cls_to_doc)"
    )
    ax.set_ylabel("Mean nDCG@10")
    plt.xticks(rotation=45)
    plt.ylim(0.48, 0.68)
    plt.tight_layout()
    plt.show()

    # Display the grouped data
    display(grouped)

In [ ]:
# Plot impact of each boolean parameter separately

# Create subplots for each boolean parameter
fig, axes = plt.subplots(1, 3, figsize=(10, 5))
fig.suptitle(
    "Impact of Each Boolean Parameter on Mean nDCG@10 ({dataset})".format(
        dataset=dataset
    )
)

grouped_dfs = []
for ax, param in zip(axes, bool_cols):
    grouped = (
        in_domain_df.groupby(param)
        .agg(mean_nDCG=("nDCG@10_mean", "mean"), mean_var=("nDCG@10_var", "mean"))
        .reset_index()
    )
    grouped["std_nDCG"] = np.sqrt(grouped["mean_var"])

    sns.barplot(data=grouped, x=param, y="mean_nDCG", ax=ax)
    ax.errorbar(
        x=range(len(grouped)),
        y=grouped["mean_nDCG"],
        yerr=grouped["std_nDCG"],
        fmt="none",
        c="black",
        capsize=3,
        elinewidth=1,
        alpha=0.8,
    )
    ax.set_title(f"Impact of {param}")
    ax.set_xlabel(param)
    ax.set_ylabel("Mean nDCG@10")
    ax.set_ylim(0.48, 0.53)

    grouped_dfs.append(grouped)

plt.tight_layout()
plt.show()

# Other results

In [ ]:
# Main table
import pandas as pd

xps = [
    "core_base_paper",
    "core_mini",
]


def get_tag(tagspath: str, regex: str) -> dict:
    match = re.search(regex, tagspath)
    if match:
        return match.group(1)
    return None


raw_dfs = []
for xp in xps:
    xp_path = xps_root / xp / "dry-run"
    raw_dfs.append(process_results(get_results(xp_path)))

df = pd.concat(raw_dfs)


# Define regex patterns for tags
tags = {
    "base": r"base=(.*)_learner",
    "loss": r"\.loss=(.*)",
}

# Split "scorer" into separate columns using the regex patterns
for tag_name, pattern in tags.items():
    df[tag_name] = df["scorer"].apply(lambda x: get_tag(x, pattern))

datasets = list(df["dataset"].unique())
print(f"{len(datasets)} unique datasets : {datasets}")
print(f"unique losses : {list(df['loss'].unique())}")
print(f"Backbones : {list(df['base'].unique())}")
df.describe()

# Figure 1

In [ ]:
import pandas as pd

# Create a list to store the new summary dataframes
summary_dfs = []

for name, datasets in aggregations.items():
    agg_result = get_aggregated_df(df, datasets=datasets, name=name)
    summary_dfs.append(agg_result)

# Combine all aggregations into one single new table
final_summary_df = pd.concat(summary_dfs, ignore_index=True)

display(final_summary_df)

In [ ]:
## Mapping and order

loss_order = list(loss_names.keys())  # Internal keys for data filtering
backbone_order = list(backbone_names.keys())
clean_labels = list(loss_names.values())  # Pretty names for the legend


def Histogram_backbonesXLosses(final_summary_df, dataset_to_plot):
    ## 2. Filter and Prepare Data
    df_plot = final_summary_df[final_summary_df["dataset"] == dataset_to_plot].copy()
    df_plot = df_plot[
        (df_plot["loss"].isin(loss_order)) & (df_plot["base"].isin(backbone_order))
    ]

    # Convert 'loss' to a categorical type with the specified order
    df_plot["loss"] = pd.Categorical(
        df_plot["loss"], categories=loss_order, ordered=True
    )

    # Sorting is now based on 'base' then your custom 'loss' order
    df_plot["loss"] = pd.Categorical(
        df_plot["loss"], categories=loss_order, ordered=True
    )
    df_plot["base"] = pd.Categorical(
        df_plot["base"], categories=backbone_order, ordered=True
    )
    df_plot = df_plot.sort_values(["base", "loss"])
    df_plot["nDCG@10_std"] = np.sqrt(df_plot["nDCG@10_var"])

    ## 3. Visualization
    fig, ax = plt.subplots(figsize=(13, 4))
    sns.set_theme(style="whitegrid", font="serif")
    # 4. Visualization

    barplot = sns.barplot(
        data=df_plot,
        x="base",
        y="nDCG@10_mean",
        hue="loss",
        hue_order=loss_order,
        palette="tab10",  # Grayscale is safer for publication
        edgecolor="black",
        linewidth=0.8,
        ax=ax,
    )

    # 5. Add Hatching (Patterns) for better visibility in print
    # hatches = ['///', '...', 'xxx', '---', '\\\\', 'OO']
    # for i, this_bar_group in enumerate(ax.containers):
    #     for bar in this_bar_group:
    #         bar.set_hatch(hatches[i % len(hatches)])

    # 6. Error Bars
    for loss_key, bar_group in zip(loss_order, ax.containers):
        subset = df_plot[df_plot["loss"] == loss_key]
        x_coords = [bar.get_x() + bar.get_width() / 2 for bar in bar_group]
        if len(x_coords) == len(subset):
            ax.errorbar(
                x=x_coords,
                y=subset["nDCG@10_mean"],
                yerr=subset["nDCG@10_std"],
                fmt="none",
                c="black",
                capsize=3,
                elinewidth=1,
                alpha=0.8,
            )

    # 7. Labels & Legend
    clean_backbones = [backbone_names[b] for b in backbone_order]
    ax.set_xticklabels(clean_backbones, rotation=45, ha="right")

    handles, _ = ax.get_legend_handles_labels()
    ax.legend(
        handles,
        list(loss_names.values()),
        title="Loss Function",
        loc="lower right",
        ncol=3,
        # bbox_to_anchor=(1.02, 1),
        frameon=True,
    )
    # Styling
    ax.set_ylim(0.2, 0.69)
    plt.ylabel(f"nDCG@10 - {dataset_to_plot}")
    ax.set_xlabel(None)
    plt.tight_layout()
    plt.savefig(
        f"./local/figures/BackbonesXLosses_{dataset_to_plot.replace(' ', '_')}.pdf"
    )

    plt.show()


for dataset_to_plot in list(aggregations.keys()):
    Histogram_backbonesXLosses(final_summary_df, dataset_to_plot)

# Tables 

## Comparison with RankDistiLLM
we extract the results from the main dataframe, but we are evaluating with the top $1000$ from `splade-v3-distillbert`, whereas they are using the top 100 from ColBERTv2

In [ ]:
reproductions = df[
    (df["base"] == "google/electra-base-discriminator")
    & (df["loss"].isin(["distillRankNET", "infoNCE_RankDistiLLM_norm_size=True"]))
    & (df["dataset"].isin(["trec2020", "trec2019"]))
].copy()

reproductions["nDCG@10_std"] = np.sqrt(100 * reproductions["nDCG@10_var"])
reproductions["nDCG@10_mean"] = 100 * reproductions["nDCG@10_mean"]
cols_to_show = ["base", "loss", "dataset", "nDCG@10_mean", "nDCG@10_std"]

# Filter for existing columns only to prevent errors
existing_cols = [c for c in cols_to_show if c in reproductions.columns]

display(reproductions[existing_cols])

## Detailed views

In [ ]:
import pandas as pd

# 1. Define the mapping for your columns to match the ID/OOD groups in the image


aggregations_table = {
    "msmarco_dev": ["msmarco_dev"],
    "trec2019": ["trec2019"],
    "trec2020": ["trec2020"],
    "BEIR13 (Semi OOD)": [
        "arguana",
        "climate_fever",
        "dbpedia",
        "fever",
        "fiqa",
        "hotpotqa",
        "nfcorpus",
        "nq",
        "quora",
        "scidocs",
        "scifact",
        "touche",
        "trec_covid",
    ],
    "Lotte-S": [
        "lotte_lifestyle",
        "lotte_recreation",
        "lotte_science",
        "lotte_technology",
        "lotte_writing",
    ],
    "robust04": ["robust04"],
}


all_relevant_datasets = aggregations_table.keys()

In [ ]:
df_latex = df.copy()

for name, datasets in aggregations_table.items():
    if len(datasets):
        agg_df = get_aggregated_df(df_latex, datasets=datasets, name=name)
        df_latex = pd.concat([df_latex, agg_df])

df_latex = df_latex[
    (df_latex["dataset"].isin(all_relevant_datasets))
    & ~(df_latex["loss"].isin(ignore_losses))
]
df_latex["loss"] = df_latex["loss"].apply(lambda x: loss_names[x])
df_latex["dataset"] = df_latex["dataset"].apply(lambda x: DATASET_TO_ABB.get(x, x))


def format_latex_backbone(x):
    name = backbone_names.get(x, x)  # Fallback to x if key not found
    return rf"\textbf{{{name}}}"


# 2. Filter and Pivot the dataframe
# Assuming 'scorer' contains 'MiniLM'/'BERT' and 'loss' contains 'BCE'/'ADR-MSE'
# We will use 'nDCG@10_mean' as the value to fill the cells, change if needed.
pivot_df = df_latex.pivot_table(
    index=["base", "loss"], columns="dataset", values="nDCG@10_mean"
)
pivot_df = pivot_df.reindex(index=backbone_names.keys(), level=0)
pivot_df.index = pivot_df.index.set_levels(
    [format_latex_backbone(x) for x in pivot_df.index.levels[0]], level=0
)

# 3. Reorder columns to match the ID/OOD layout exactly
pivot_df = pivot_df[[DATASET_TO_ABB.get(x, x) for x in all_relevant_datasets]]


# 2. Function to bold the max value per column
def bold_max(col):
    # Find the index of the maximum value
    idx_max = col.idxmax()

    # Format all values to 3 decimal places
    # If it's the max, wrap it in \textbf{}
    return [
        f"\\textbf{{{x:.3f}}}" if i == idx_max else f"{x:.3f}" for i, x in col.items()
    ]


# Apply the function to the whole dataframe
pivot_df = pivot_df.apply(bold_max)

# 4. Generate the LaTeX code
# Multi-indexing in Pandas automatically creates the multi-row structure for 'scorer'
latex_output = pivot_df.to_latex(
    multicolumn=True,
    multirow=True,
    caption="Evaluation Results for ID and OOD datasets",
    label="tab:results",
    column_format="llcccccc",  # Aligns Loss/Scorer and adds separators for ID/OOD
    float_format="%.3f",
)

# 5. Adding the ID/OOD header manually
# Pandas doesn't natively do the "ID" over 3 cols and "OOD" over 3 cols in one step,
# so we do a quick string replacement for the top header:
# header_replacement = (
#     " & & \multicolumn{3}{c}{ID} & \multicolumn{3}{c}{OOD} \\\\\n"
#     "Scorer & Loss & MSM & DL19 & DL20 & Beir 13 & Lotte & Robust \\\\"
# )

# Simple way to inject the double header
lines = latex_output.splitlines()
# lines[4] = header_replacement # Replaces the default header line
final_latex = "\n".join(lines).replace("\multirow[t]", "\multirow[c]")

print(final_latex)


with open("./local/figures/evaluation_table.tex", "w") as f:
    f.write(final_latex)

print("File 'evaluation_table.tex' has been created successfully.")
display(pivot_df)

# Figure 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from pathlib import Path


def plot_ettin_scaling_refined(
    df, selected_losses=None, selected_datasets=None, loss_names=None
):
    # Ensure mapping dict exists
    loss_names = loss_names or {}

    # 1. Filter and Prepare Data
    plot_df = df[df["base"].str.contains("ettin", na=False)].copy()
    plot_df["model_size"] = plot_df["base"].str.extract(r"-(\d+)m").astype(int)
    plot_df["nDCG@10_std"] = np.sqrt(plot_df["nDCG@10_var"])

    # Apply user selections
    if selected_losses:
        plot_df = plot_df[plot_df["loss"].isin(selected_losses)]
    if selected_datasets:
        plot_df = plot_df[plot_df["dataset"].isin(selected_datasets)]

    plot_df = plot_df.sort_values("model_size")

    # 2. Setup Aesthetic Mappings
    sns.set_theme(style="whitegrid", font="serif")
    fig, ax = plt.subplots(figsize=(8, 6))

    # Sticky colors for datasets
    unique_ds = sorted(plot_df["dataset"].unique())
    palette = dict(zip(unique_ds, sns.color_palette("bright", len(unique_ds))))

    # Markers for losses
    unique_ls = sorted(plot_df["loss"].unique())
    markers = ["o", "s", "^", "D", "p", "H", "X"]
    marker_map = dict(zip(unique_ls, markers[: len(unique_ls)]))

    # 3. Plotting with Groupby
    for (ds, ls), subset in plot_df.groupby(["dataset", "loss"]):
        ax.errorbar(
            x=subset["model_size"],
            y=subset["nDCG@10_mean"],
            yerr=subset["nDCG@10_std"],
            color=palette[ds],
            marker=marker_map[ls],
            linestyle="-",
            linewidth=1.5,
            markersize=7,
            capsize=3,
            alpha=0.85,
        )

    # 4. Legend Construction (Using loss_names)
    # Part A: Dataset Colors
    ds_handles = [Line2D([0], [0], color=palette[d], lw=3, label=d) for d in unique_ds]

    # Part B: Loss Markers (Prettified)
    ls_handles = [
        Line2D(
            [0],
            [0],
            marker=marker_map[l],
            color="gray",
            ls="",
            markersize=8,
            label=loss_names.get(l, l),
        )
        for l in unique_ls
    ]

    # Combine with a spacer
    spacer = Line2D([0], [0], color="none", label="")
    ax.legend(
        handles=ds_handles + [spacer] + ls_handles,
        # bbox_to_anchor=(1.02, 1),
        loc="lower right",
        title="Datasets & Objectives",
        frameon=True,
    )

    # 5. Final Polish
    ax.set(
        xscale="log",
        xlabel="Model Size (Millions of Parameters)",
        ylabel="nDCG@10",
    )

    # Format X-axis to show actual size numbers
    ticks = sorted(plot_df["model_size"].unique())
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(plt.ScalarFormatter())

    plt.tight_layout()
    plt.savefig("./local/figures/scaling_law_ettin.pdf")
    plt.show()


# Select what you want to see
plot_ettin_scaling_refined(
    final_summary_df,
    selected_losses=[
        "marginMSE",
        #  "distillRankNET",
        "infoNCE_RankDistiLLM_norm_size=True",
    ],
    selected_datasets=aggregations.keys(),
    loss_names=loss_names,
)